# 03 — Positional Embedding 可視化

モデルが「どこにあるか」を学習した**位置埋め込み**の内部構造を可視化する。

- **コサイン類似度マトリクス**: 65 トークン間の類似度（隣接パッチが近ければ位置が学習できている）
- **各位置の類似度マップ**: ある 1 つのパッチと他パッチとの類似度を 8×8 マップで表示
- **PCA 2D プロット**: 256 次元の位置埋め込みを 2D に圧縮し、格子状に並ぶかを確認


In [ ]:
import sys, pathlib
# notebooks/ の親（vit/）を Python パスに追加して vit パッケージを import できるようにする
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import matplotlib
import matplotlib.font_manager as _fm
# macOS 標準の日本語フォントを自動検出して設定（なければ DejaVu にフォールバック）
_jp = next((f.name for f in _fm.fontManager.ttflist if "Hiragino Sans" == f.name), None)
if _jp:
    matplotlib.rcParams["font.family"] = _jp
matplotlib.rcParams["axes.unicode_minus"] = False  # マイナス記号の文字化け防止

import torch
import torchvision
import torchvision.transforms as T
import matplotlib.pyplot as plt
from viz_utils import (
    CIFAR10_CLASSES, CIFAR10_MEAN, CIFAR10_STD,
    denormalize, to_hwc, show_image, show_patches,
    overlay_attention, attention_rollout,
)
from vit import ViT

MODEL_PATH  = "../best_vit.pth"
DATA_ROOT   = "../data"
PATCH_SIZE  = 4
EMBED_DIM   = 256
DEPTH       = 6
NUM_HEADS   = 8
MLP_DIM     = 512
model = ViT(
    image_size=32, patch_size=PATCH_SIZE, num_classes=10,
    embed_dim=EMBED_DIM, depth=DEPTH, num_heads=NUM_HEADS,
    mlp_dim=MLP_DIM, dropout=0.1, emb_dropout=0.1,
)
model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
model.eval()
print(f"モデルロード完了  パラメータ数: {model.num_parameters():,}")


## 位置埋め込みを取り出す

`model.pos_embed` は `(1, 65, 256)` のパラメータ。
- `pos_embed[0, 0]`    : CLS token の位置埋め込み
- `pos_embed[0, 1:65]` : パッチ 0–63 の位置埋め込み（row-major, 8×8 順）


In [ ]:
pos_embed = model.pos_embed[0].detach()  # (65, 256)

cls_pos   = pos_embed[0]     # (256,)   CLS
patch_pos = pos_embed[1:]    # (64, 256) パッチ

print(f"pos_embed: {tuple(pos_embed.shape)}")
print(f"  CLS  pos: {tuple(cls_pos.shape)},  norm = {cls_pos.norm():.4f}")
print(f"  patch pos (mean norm): {patch_pos.norm(dim=-1).mean():.4f}")


## コサイン類似度マトリクス

全 65 トークンの位置埋め込み間のコサイン類似度を 65×65 のヒートマップで表す。
- 対角が 1.0 になるのは自分自身との類似度
- **隣接パッチが近い色**なら、位置情報が空間的に学習できている証拠


In [ ]:
import torch.nn.functional as F

# コサイン類似度: (65, 65)
pos_norm = F.normalize(pos_embed, dim=-1)
sim = (pos_norm @ pos_norm.T).numpy()   # (65, 65)

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(sim, cmap="RdYlBu_r", vmin=-1, vmax=1, interpolation="nearest")
ax.set_title("位置埋め込みのコサイン類似度マトリクス (65×65)\n"
             "index 0 = CLS、index 1–64 = patches (row-major)", fontsize=11)
ax.set_xlabel("Token index")
ax.set_ylabel("Token index")
# CLS 境界を強調
ax.axhline(0.5, color="white", lw=1.5)
ax.axvline(0.5, color="white", lw=1.5)
ax.text(0, 0, "CLS", ha="center", va="center", fontsize=8, color="white", fontweight="bold")
plt.colorbar(im, fraction=0.04, pad=0.03)
plt.tight_layout()
plt.show()

# 上位類似ペアを確認（CLS を除く）
sim_patch = sim[1:, 1:]   # (64, 64)
print("パッチ間コサイン類似度の統計:")
print(f"  mean = {sim_patch.mean():.4f}")
print(f"  std  = {sim_patch.std():.4f}")
print(f"  隣接パッチ（左右 1 ステップ）の平均類似度 ≈ "
      f"{sim_patch[range(0, 56), range(1, 57)].mean():.4f}")


## 各パッチ位置の類似度マップ

あるパッチと他の全パッチとのコサイン類似度を 8×8 マップとして可視化する。
**空間的に連続した構造**があれば位置埋め込みが正しく学習できている。


In [ ]:
# 4 つの代表的な位置を選ぶ
positions = {
    "左上 (0,0)":   0,
    "右上 (0,7)":   7,
    "中央 (3,3)":  27,
    "右下 (7,7)":  63,
}

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, (label, idx) in zip(axes, positions.items()):
    sim_row = sim[idx + 1, 1:].reshape(8, 8)   # +1 は CLS offset
    im = ax.imshow(sim_row, cmap="RdYlBu_r", vmin=-0.5, vmax=1.0,
                   interpolation="nearest")
    # 基準パッチの位置に × 印
    row, col = divmod(idx, 8)
    ax.plot(col, row, "wx", markersize=12, markeredgewidth=2.5)
    ax.set_title(f"{label}\n(patch {idx})", fontsize=10)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.06)

fig.suptitle("各パッチと他パッチとのコサイン類似度マップ\n"
             "（×=基準パッチ、周囲ほど明るい = 空間的な位置が学習できている）",
             fontsize=12)
plt.tight_layout()
plt.show()


## PCA で 2D 可視化

64 パッチの位置埋め込み（256 次元）を PCA で 2 次元に圧縮する。
うまく学習できていれば **8×8 の格子状パターン**が現れるはず。


In [ ]:
# PyTorch の pca_lowrank を使う（scikit-learn 不要）
U, S, V = torch.pca_lowrank(patch_pos, q=2)
coords = (patch_pos @ V).numpy()  # (64, 2)

fig, ax = plt.subplots(figsize=(7, 7))

# 8×8 グリッドの行・列ごとに色を付ける
row_ids = torch.arange(64) // 8   # 0–7
col_ids = torch.arange(64) %  8   # 0–7

scatter = ax.scatter(coords[:, 0], coords[:, 1],
                     c=row_ids.numpy(), cmap="tab10",
                     s=120, zorder=3)
# パッチ番号のラベル
for idx, (x, y) in enumerate(coords):
    ax.annotate(str(idx), (x, y), fontsize=7,
                ha="center", va="center", color="white", fontweight="bold")

# 格子の辺（行が同じパッチを線でつなぐ）
for r in range(8):
    row_mask = (row_ids == r).numpy()
    row_coords = coords[row_mask]
    # 列順にソート
    order = col_ids[row_ids == r].numpy().argsort()
    ax.plot(row_coords[order, 0], row_coords[order, 1],
            "-", color="gray", alpha=0.4, lw=0.8)

plt.colorbar(scatter, ax=ax, label="行インデックス (0=上、7=下)")
ax.set_title("位置埋め込みの PCA 2D 可視化\n"
             "（色=行, 数字=パッチ番号, 格子状に並ぶほど位置が学習されている）",
             fontsize=11)
ax.set_xlabel("PC 1")
ax.set_ylabel("PC 2")
plt.tight_layout()
plt.show()

explained = (S[:2] ** 2 / (S ** 2).sum() * 100)
print(f"PC1 説明分散率: {explained[0]:.1f}%")
print(f"PC2 説明分散率: {explained[1]:.1f}%")
print(f"PC1+PC2 合計  : {explained.sum():.1f}%")


## まとめ

| 観察 | 意味 |
|---|---|
| 類似度マトリクスで隣接パッチが高い相関 | モデルが空間的な近さを学習できている |
| 類似度マップで中心からの距離勾配 | 位置情報が連続的 |
| PCA プロットが格子状 | 埋め込みが 2D 空間の構造を保持している |
| CLS が全パッチと独立した低類似度 | CLS は特殊な「集約用」トークンとして機能している |
